# Test H1 — real agents on Hyperliquid: collector and (from 10 December 2026) the registered analysis

Registered in `thesis/checks/prereg/H1_hyperliquid.json` **before any trade was recorded**: the data do not exist yet.
Hyperliquid publishes every trade with the buyer's and the seller's wallet address. This notebook records them for 30 markets to your Google Drive (`MyDrive/h1_data`), and from 10 December 2026 runs the registered analysis **once**.

**How to collect.** Run cells 1–3. Cell 3 records for up to 11.5 hours, then stops by itself. **Restart it about twice a day** (run cells 1–3 again): the universe and the files on Drive are reused and merged, so restarts and overlaps are safe. Days with less than 95% connection time are not used, so gaps cost days, not correctness. Collect until at least 10 December 2026 (ideally 40+ days).

**Do not open or inspect the trade files** before the analysis: the test is registered so that nobody has seen the data. Cell 4 shows only how many hours were recorded.

**Analysis.** From 10 December 2026, run cells 1, 2 and 5. It opens only if at least 20 markets have 30 valid days (on 31 January 2027 it runs regardless). Send `h1_result.zip` in the chat.

In [ ]:
# 1. Registered code, Drive, and the WebSocket library
import os, subprocess, sys, time
REPO = "/content/Fianince"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/Pratikn03/Fianince.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "websockets"], check=True)
sys.path.insert(0, f"{REPO}/thesis/evidence")
from google.colab import drive
for attempt in range(3):
    try:
        drive.mount("/content/drive", force_remount=True, timeout_ms=300000); break
    except ValueError as e:
        print("mount attempt", attempt + 1, "failed:", e); time.sleep(10)
else:
    raise SystemExit("Drive did not mount; tell Claude")
OUT = "/content/drive/MyDrive/h1_data"; os.makedirs(OUT, exist_ok=True); print("Drive OK:", OUT)

In [ ]:
# 2. Check the registration and that Hyperliquid's public API answers from here (no VPN or proxy: if it is blocked, stop and tell Claude)
import json, importlib
import h1_hyperliquid as C; importlib.reload(C)
spec, sha, commit = C.gate()
print("specification sha256:", sha, "| registered in commit:", commit)
meta, ctxs = C.http_post({"type": "metaAndAssetCtxs"})
print("Hyperliquid API OK:", len(meta["universe"]), "perpetuals listed")

In [ ]:
# 3. COLLECT for up to 11.5 hours (the first run draws and saves the 30-market universe before saving any trade)
n = await C.collect(OUT, hours=11.5)

In [ ]:
# 4. Status: recorded hours and connection coverage only (does not read the trades)
from pathlib import Path
print("universe:", json.loads((Path(OUT) / "H1_universe.json").read_text())["symbols"])
for d in C.collected_dates(OUT):
    files = len(list((Path(OUT) / "trades").glob(f"*/{d}/*.csv.gz")))
    print(d, f"coverage {100 * C.coverage(OUT, d):5.1f}%", f"hourly files {files}")

In [ ]:
# 5. ANALYSIS (once, from 2026-12-10). The counter is kept on Drive so that it can never run twice.
import shutil, zipfile
REPO_COUNTER = Path(REPO) / "thesis/checks/prereg/H1_unlock.json"; DRIVE_COUNTER = Path(OUT) / REPO_COUNTER.name
if not DRIVE_COUNTER.exists():
    shutil.copy(REPO_COUNTER, DRIVE_COUNTER)
C.COUNTER = DRIVE_COUNTER
res = C.run(OUT)
shutil.copy(DRIVE_COUNTER, REPO_COUNTER); shutil.copy(Path(OUT) / "H1_universe.json", Path(REPO) / "thesis/evidence/H1_universe.json")
FILES = ["thesis/checks/prereg/H1_unlock.json", "thesis/evidence/H1_universe.json", "thesis/evidence/H1_results.json", "thesis/checks/H1_numbers.tex"]
with zipfile.ZipFile("/content/h1_result.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(f"{REPO}/{f}", f)
from google.colab import files
files.download("/content/h1_result.zip")